Montly

In [11]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline

# ============================================================
# STEP 0 — Paths
# ============================================================

BASE = "data/macro_processed"

paths = {
    "sp500":        f"{BASE}/other/sp500_processed.csv",
    "tbill_3m":     f"{BASE}/other/3m_yield_processed.csv",

    # Inflation
    "cpi":          f"{BASE}/inflation/cpi_processed.csv",
    "pce":          f"{BASE}/inflation/PCE_price_index_processed.csv",
    "ppi":          f"{BASE}/inflation/PPI_inflation_processed.csv",

    # Growth
    "indprod":      f"{BASE}/ec_growth/industrial_production_processed.csv",
    "retail_sales": f"{BASE}/ec_growth/retail_sales_processed.csv",
    "inventories":  f"{BASE}/ec_growth/tot_business_inventories_processed.csv",
    "export_px":    f"{BASE}/ec_growth/export_price_index_processed.csv",
    "import_px":    f"{BASE}/ec_growth/import_price_index_processed.csv",
    "unemp":        f"{BASE}/ec_growth/unemployment_processed.csv",

    # Money/policy
    "m2":           f"{BASE}/mon_policy/m2_real_money_supply_processed.csv",
    "fedfunds":     f"{BASE}/mon_policy/fedfunds_processed.csv",
    "discount_rate":f"{BASE}/mon_policy/fed_reserve_discount_rate_processed.csv",

    # Rates, spreads
    "spread_10y_2y":f"{BASE}/mkt_vol/10y_2y_spread_processed.csv",
    "nat_fin_cond": f"{BASE}/mkt_vol/nat_fin_condition_indx_processed.csv",
    "nasdaq_vol":   f"{BASE}/mkt_vol/nasdaq_vol_indx_processed.csv",
    "hy_spread":    f"{BASE}/other/bofa_highyield_spread_processed.csv",
    "y2":           f"{BASE}/other/2y_yield_processed.csv",
    "y3m":          f"{BASE}/other/3m_yield_processed.csv",
    "y10":          f"{BASE}/other/10y_yield_processed.csv",
}

# ============================================================
# STEP 1 — Monthly geometric EMRP (no quarterly needed)
# ============================================================

sp = pd.read_csv(paths["sp500"], parse_dates=["date"]).set_index("date").sort_index()
sp["sp500_geo_m"] = sp["pct_change_mom"] / 100.0
sp_m = sp.resample("M").last()[["sp500_geo_m"]]

rf = pd.read_csv(paths["tbill_3m"], parse_dates=["date"]).set_index("date").sort_index()
rf["r_daily"] = (rf["value"]/100.0)/252.0
rf_m = rf["r_daily"].resample("M").apply(lambda x: (1 + x).prod() - 1)
rf_m = rf_m.to_frame("rf_geo_m")

m = sp_m.join(rf_m, how="inner")
m["EMRP_geo_m"] = m["sp500_geo_m"] - m["rf_geo_m"]

m["EMRP_next_m"] = m["EMRP_geo_m"].shift(-1)

# ============================================================
# STEP 2 — Load monthly macro + MoM % changes + level vars + lag
# ============================================================

def load_monthly(path, name):
    df = pd.read_csv(path, parse_dates=["date"]).set_index("date").sort_index()
    return df[[ "value" ]].rename(columns={"value": name}).resample("M").last()

def safe_pct_change(df, lag=1):
    out = df.replace(0, np.nan).pct_change(lag) * 100
    out = out.replace([np.inf,-np.inf], np.nan)
    return out

# Load monthly series
cpi_m  = load_monthly(paths["cpi"], "cpi")
pce_m  = load_monthly(paths["pce"], "pce")
ppi_m  = load_monthly(paths["ppi"], "ppi")

ind_m  = load_monthly(paths["indprod"], "indprod")
ret_m  = load_monthly(paths["retail_sales"], "retail_sales")
inv_m  = load_monthly(paths["inventories"], "inventories")
exp_m  = load_monthly(paths["export_px"], "export_px")
imp_m  = load_monthly(paths["import_px"], "import_px")

unemp_m  = load_monthly(paths["unemp"], "unemp_rate")
fedf_m   = load_monthly(paths["fedfunds"], "fedfunds")
disc_m   = load_monthly(paths["discount_rate"], "discount_rate")

spread_m = load_monthly(paths["spread_10y_2y"], "spread_10y_2y")
natfin_m = load_monthly(paths["nat_fin_cond"], "nat_fin_cond")
nasd_m   = load_monthly(paths["nasdaq_vol"], "nasdaq_vol")
hy_m     = load_monthly(paths["hy_spread"], "hy_spread")
y2_m     = load_monthly(paths["y2"], "y2")
y3m_m    = load_monthly(paths["y3m"], "y3m")
y10_m    = load_monthly(paths["y10"], "y10")
m2_m     = load_monthly(paths["m2"], "m2")

macro_levels_m = pd.concat([
    cpi_m,pce_m,ppi_m,
    ind_m,ret_m,inv_m,
    exp_m,imp_m,
    unemp_m,m2_m,fedf_m,disc_m,
    spread_m,natfin_m,nasd_m,hy_m,
    y2_m,y3m_m,y10_m
], axis=1)

macro_m = safe_pct_change(macro_levels_m).add_suffix("_mom")

level_vars = [
    "unemp_rate","fedfunds","discount_rate","spread_10y_2y",
    "nat_fin_cond","nasdaq_vol","hy_spread","y2","y3m","y10"
]

for col in level_vars:
    macro_m[col] = macro_levels_m[col]
    macro_m.drop(columns=[col+"_mom"], errors="ignore", inplace=True)

# LAG ALL MACRO VARIABLES BY ONE MONTH:
macro_m_lagged = macro_m.shift(1)

# ============================================================
# STEP 3 — Merge monthly EMRP and lagged predictors
# ============================================================

reg_m = m.join(macro_m_lagged, how="inner").replace([np.inf,-np.inf], np.nan).dropna()
predictors_m = [c for c in reg_m.columns if c not in ["EMRP_geo_m", "EMRP_next_m"]]

print("Monthly sample size:", len(reg_m))

# ============================================================
# STEP 4 — Monthly multivariate OLS
# ============================================================

df_multi_m = reg_m[["EMRP_next_m"] + predictors_m]
y_m = df_multi_m["EMRP_next_m"]
X_m = sm.add_constant(df_multi_m[predictors_m])

model_m = sm.OLS(y_m, X_m).fit()
print("\n=== MONTHLY MULTIVARIATE OLS ===")
print(model_m.summary())

# ============================================================
# STEP 5 — Monthly univariate regressions
# ============================================================

simple_rows_m = []
for var in predictors_m:
    tmp = reg_m[["EMRP_next_m", var]].dropna()
    if tmp.shape[0] < 20:
        continue
    y_s = tmp["EMRP_next_m"]
    X_s = sm.add_constant(tmp[[var]])
    res = sm.OLS(y_s, X_s).fit()

    simple_rows_m.append({
        "variable": var,
        "coef": res.params[var],
        "t_stat": res.tvalues[var],
        "p_value": res.pvalues[var],
        "R_squared": res.rsquared,
        "n_obs": int(res.nobs),
        "sign": "positive" if res.params[var] > 0 else "negative"
    })

simple_m_df = pd.DataFrame(simple_rows_m).sort_values("p_value")
print("\n=== MONTHLY UNIVARIATE REGRESSIONS ===")
print(simple_m_df.to_string(index=False))

# ============================================================
# STEP 6 — MONTHLY LASSO (FORCE 6–10 VARIABLES)
# ============================================================

X_full_m = reg_m[predictors_m].values
y_full_m = reg_m["EMRP_next_m"].values

alphas = np.logspace(-3, 0, 100)
tol = 1e-6
best_alpha = None
best_coefs = None
best_k = None

for a in sorted(alphas, reverse=True):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", Lasso(alpha=a, max_iter=50000))
    ])
    pipe.fit(X_full_m, y_full_m)
    coefs = pipe.named_steps["lasso"].coef_
    k = np.sum(np.abs(coefs) > tol)
    if 6 <= k <= 10:
        best_alpha = a
        best_coefs = coefs
        best_k = k
        break

if best_alpha is None:
    a = alphas.min()
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", Lasso(alpha=a, max_iter=50000))
    ])
    pipe.fit(X_full_m, y_full_m)
    best_alpha = a
    best_coefs = pipe.named_steps["lasso"].coef_
    best_k = np.sum(np.abs(best_coefs) > tol)

selected_vars_m = [p for p, c in zip(predictors_m, best_coefs) if abs(c) > tol]

print("\n=== MONTHLY LASSO (6–10 VARIABLES) ===")
print("alpha:", best_alpha)
print("nonzero:", best_k)
print("selected:", selected_vars_m)

print("\nLASSO coefficients:")
for p, c in sorted(zip(predictors_m, best_coefs), key=lambda x: -abs(x[1])):
    if abs(c) > tol:
        print(f"{p:20s} {c:+.5f}")

# ============================================================
# STEP 7 — Reduced OLS (post-LASSO)
# ============================================================

df_red_m = reg_m[["EMRP_next_m"] + selected_vars_m].dropna()
y_rm = df_red_m["EMRP_next_m"]
X_rm = sm.add_constant(df_red_m[selected_vars_m])

model_red_m = sm.OLS(y_rm, X_rm).fit()
print("\n=== MONTHLY REDUCED OLS (POST-LASSO) ===")
print(model_red_m.summary())

/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_51876/3541480497.py:53: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  sp_m = sp.resample("M").last()[["sp500_geo_m"]]
/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_51876/3541480497.py:57: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  rf_m = rf["r_daily"].resample("M").apply(lambda x: (1 + x).prod() - 1)
/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_51876/3541480497.py:71: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  return df[[ "value" ]].rename(columns={"value": name}).resample("M").last()
/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_51876/3541480497.py:71: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  return df[[ "value" ]].rename(columns={"value": name}).resample("M").

Monthly sample size: 247

=== MONTHLY MULTIVARIATE OLS ===
                            OLS Regression Results                            
Dep. Variable:            EMRP_next_m   R-squared:                       0.178
Model:                            OLS   Adj. R-squared:                  0.105
Method:                 Least Squares   F-statistic:                     2.444
Date:                Wed, 26 Nov 2025   Prob (F-statistic):           0.000812
Time:                        00:01:35   Log-Likelihood:                 452.31
No. Observations:                 247   AIC:                            -862.6
Df Residuals:                     226   BIC:                            -788.9
Df Model:                          20                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------

/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_51876/3541480497.py:71: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  return df[[ "value" ]].rename(columns={"value": name}).resample("M").last()
/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_51876/3541480497.py:71: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  return df[[ "value" ]].rename(columns={"value": name}).resample("M").last()
/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_51876/3541480497.py:74: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.replace(0, np.nan).pct_change(lag) * 100


Quarterly  

In [10]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline

# ============================================================
# STEP 0 — Paths
# ============================================================

BASE = "data/macro_processed"

paths = {
    "sp500":        f"{BASE}/other/sp500_processed.csv",
    "tbill_3m":     f"{BASE}/other/3m_yield_processed.csv",

    # Inflation indices (levels)
    "cpi":          f"{BASE}/inflation/cpi_processed.csv",
    "pce":          f"{BASE}/inflation/PCE_price_index_processed.csv",
    "ppi":          f"{BASE}/inflation/PPI_inflation_processed.csv",

    # Growth indicators (levels)
    "indprod":      f"{BASE}/ec_growth/industrial_production_processed.csv",
    "retail_sales": f"{BASE}/ec_growth/retail_sales_processed.csv",
    "inventories":  f"{BASE}/ec_growth/tot_business_inventories_processed.csv",
    "export_px":    f"{BASE}/ec_growth/export_price_index_processed.csv",
    "import_px":    f"{BASE}/ec_growth/import_price_index_processed.csv",
    "unemp":        f"{BASE}/ec_growth/unemployment_processed.csv",

    # Money and policy
    "m2":           f"{BASE}/mon_policy/m2_real_money_supply_processed.csv",
    "fedfunds":     f"{BASE}/mon_policy/fedfunds_processed.csv",
    "discount_rate":f"{BASE}/mon_policy/fed_reserve_discount_rate_processed.csv",

    # Yields and spreads
    "spread_10y_2y":f"{BASE}/mkt_vol/10y_2y_spread_processed.csv",
    "nat_fin_cond": f"{BASE}/mkt_vol/nat_fin_condition_indx_processed.csv",
    "nasdaq_vol":   f"{BASE}/mkt_vol/nasdaq_vol_indx_processed.csv",
    "hy_spread":    f"{BASE}/other/bofa_highyield_spread_processed.csv",
    "y2":           f"{BASE}/other/2y_yield_processed.csv",
    "y3m":          f"{BASE}/other/3m_yield_processed.csv",
    "y10":          f"{BASE}/other/10y_yield_processed.csv",
}

# ============================================================
# STEP 1 — Monthly geometric EMRP, then quarterly aggregation
# ============================================================

sp = pd.read_csv(paths["sp500"], parse_dates=["date"]).set_index("date").sort_index()
sp["sp500_geo_m"] = sp["pct_change_mom"] / 100.0
sp_m = sp.resample("M").last()[["sp500_geo_m"]]

rf = pd.read_csv(paths["tbill_3m"], parse_dates=["date"]).set_index("date").sort_index()
rf["r_daily"] = (rf["value"] / 100.0) / 252.0
rf_m = rf["r_daily"].resample("M").apply(lambda x: (1 + x).prod() - 1)
rf_m = rf_m.to_frame("rf_geo_m")

m = sp_m.join(rf_m, how="inner")
m["EMRP_geo_m"] = m["sp500_geo_m"] - m["rf_geo_m"]

emrp_q = (1 + m["EMRP_geo_m"]).resample("Q").prod() - 1
emrp_q = emrp_q.to_frame("EMRP_q")
emrp_q["EMRP_next_q"] = emrp_q["EMRP_q"].shift(-1)

# ============================================================
# STEP 2 — Macro predictors (QoQ) + levels + 1Q lag
# ============================================================

def load_monthly(path, date_col="date", value_col="value", name=None):
    df = pd.read_csv(path, parse_dates=[date_col]).set_index(date_col).sort_index()
    if name is None:
        name = value_col
    return df[[value_col]].rename(columns={value_col: name}).resample("M").last()

def safe_pct_change(df, lag=1):
    out = df.replace(0, np.nan).pct_change(lag) * 100.0
    out = out.replace([np.inf, -np.inf], np.nan)
    return out

# Load all monthly macro series
cpi_m  = load_monthly(paths["cpi"],          name="cpi")
pce_m  = load_monthly(paths["pce"],          name="pce")
ppi_m  = load_monthly(paths["ppi"],          name="ppi")
ind_m  = load_monthly(paths["indprod"],      name="indprod")
ret_m  = load_monthly(paths["retail_sales"], name="retail_sales")
inv_m  = load_monthly(paths["inventories"],  name="inventories")
exp_m  = load_monthly(paths["export_px"],    name="export_px")
imp_m  = load_monthly(paths["import_px"],    name="import_px")
m2_m   = load_monthly(paths["m2"],           name="m2")

unemp_m  = load_monthly(paths["unemp"],      name="unemp_rate")
fedf_m   = load_monthly(paths["fedfunds"],   name="fedfunds")
disc_m   = load_monthly(paths["discount_rate"], name="discount_rate")
spread_m = load_monthly(paths["spread_10y_2y"], name="spread_10y_2y")
natfin_m = load_monthly(paths["nat_fin_cond"], name="nat_fin_cond")
nasd_m   = load_monthly(paths["nasdaq_vol"], name="nasdaq_vol")
hy_m     = load_monthly(paths["hy_spread"],  name="hy_spread")
y2_m     = load_monthly(paths["y2"],         name="y2")
y3m_m    = load_monthly(paths["y3m"],        name="y3m")
y10_m    = load_monthly(paths["y10"],        name="y10")

macro_levels_m = pd.concat([
    cpi_m, pce_m, ppi_m,
    ind_m, ret_m, inv_m,
    exp_m, imp_m,
    unemp_m, m2_m, fedf_m, disc_m,
    spread_m, natfin_m, nasd_m, hy_m,
    y2_m, y3m_m, y10_m
], axis=1)

macro_q_levels = macro_levels_m.resample("Q").last()

macro_q = safe_pct_change(macro_q_levels).add_suffix("_qoq")

level_vars = [
    "unemp_rate","fedfunds","discount_rate","spread_10y_2y",
    "nat_fin_cond","nasdaq_vol","hy_spread","y2","y3m","y10"
]

for col in level_vars:
    macro_q[col] = macro_q_levels[col]
    macro_q.drop(columns=[col + "_qoq"], errors="ignore", inplace=True)

# LAG ALL PREDICTORS BY ONE QUARTER
macro_q_lagged = macro_q.shift(1)

# ============================================================
# STEP 3 — Merge return + lagged macro
# ============================================================

reg_q = emrp_q.join(macro_q_lagged, how="inner").replace([np.inf, -np.inf], np.nan).dropna()
predictors = [c for c in reg_q.columns if c not in ["EMRP_q", "EMRP_next_q"]]

print("Quarterly sample size:", len(reg_q))

# ============================================================
# STEP 4 — Multivariate quarterly OLS
# ============================================================

df_multi_q = reg_q[["EMRP_next_q"] + predictors]
y_q = df_multi_q["EMRP_next_q"]
X_q = sm.add_constant(df_multi_q[predictors])

model_q = sm.OLS(y_q, X_q).fit()
print("\n=== QUARTERLY MULTIPLE OLS ===")
print(model_q.summary())

# ============================================================
# STEP 5 — Univariate quarterly regressions
# ============================================================

simple_rows_q = []
for var in predictors:
    tmp = reg_q[["EMRP_next_q", var]].dropna()
    if tmp.shape[0] < 10:
        continue
    y_s = tmp["EMRP_next_q"]
    X_s = sm.add_constant(tmp[[var]])
    res = sm.OLS(y_s, X_s).fit()

    simple_rows_q.append({
        "variable": var,
        "coef": res.params[var],
        "t_stat": res.tvalues[var],
        "p_value": res.pvalues[var],
        "R_squared": res.rsquared,
        "n_obs": int(res.nobs),
        "sign": "positive" if res.params[var] > 0 else "negative"
    })

simple_q_df = pd.DataFrame(simple_rows_q).sort_values("p_value")
print("\n=== QUARTERLY UNIVARIATE REGRESSIONS ===")
print(simple_q_df.to_string(index=False))

# ============================================================
# STEP 6 — LASSO (FORCE 6–10 VARIABLES)
# ============================================================

X_full_q = reg_q[predictors].values
y_full_q = reg_q["EMRP_next_q"].values

alphas = np.logspace(-3, 0, 100)
tol = 1e-6

best_alpha = None
best_coefs = None
best_k = None

for a in sorted(alphas, reverse=True):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", Lasso(alpha=a, max_iter=50000, random_state=0))
    ])
    pipe.fit(X_full_q, y_full_q)
    coefs = pipe.named_steps["lasso"].coef_
    k = np.sum(np.abs(coefs) > tol)

    if 6 <= k <= 10:
        best_alpha = a
        best_coefs = coefs
        best_k = k
        break

if best_alpha is None:
    a = alphas.min()
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", Lasso(alpha=a, max_iter=50000, random_state=0))
    ])
    pipe.fit(X_full_q, y_full_q)
    best_alpha = a
    best_coefs = pipe.named_steps["lasso"].coef_
    best_k = np.sum(np.abs(best_coefs) > tol)

selected_vars = [p for p, c in zip(predictors, best_coefs) if abs(c) > tol]

print("\n=== QUARTERLY LASSO (6–10 VARIABLES) ===")
print("alpha:", best_alpha)
print("nonzero:", best_k)
print("selected:", selected_vars)

print("\nLASSO coefficients:")
for p, c in sorted(zip(predictors, best_coefs), key=lambda x: -abs(x[1])):
    if abs(c) > tol:
        print(f"{p:20s} {c:+.5f}")

# ============================================================
# STEP 7 — Reduced OLS (post-LASSO)
# ============================================================

df_red_q = reg_q[["EMRP_next_q"] + selected_vars].dropna()
y_rq = df_red_q["EMRP_next_q"]
X_rq = sm.add_constant(df_red_q[selected_vars])

model_red_q = sm.OLS(y_rq, X_rq).fit()

print("\n=== REDUCED OLS (POST-LASSO) ===")
print(model_red_q.summary())

/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_51876/1510097216.py:53: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  sp_m = sp.resample("M").last()[["sp500_geo_m"]]
/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_51876/1510097216.py:57: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  rf_m = rf["r_daily"].resample("M").apply(lambda x: (1 + x).prod() - 1)
/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_51876/1510097216.py:63: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  emrp_q = (1 + m["EMRP_geo_m"]).resample("Q").prod() - 1
/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_51876/1510097216.py:75: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  return df[[value_col]].rename(columns={value_col: name}).resample("M").last()
/var/folder

Quarterly sample size: 83

=== QUARTERLY MULTIPLE OLS ===
                            OLS Regression Results                            
Dep. Variable:            EMRP_next_q   R-squared:                       0.411
Model:                            OLS   Adj. R-squared:                  0.245
Method:                 Least Squares   F-statistic:                     2.481
Date:                Tue, 25 Nov 2025   Prob (F-statistic):            0.00408
Time:                        23:28:33   Log-Likelihood:                 110.67
No. Observations:                  83   AIC:                            -183.3
Df Residuals:                      64   BIC:                            -137.4
Df Model:                          18                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------

/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_51876/1510097216.py:75: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  return df[[value_col]].rename(columns={value_col: name}).resample("M").last()
/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_51876/1510097216.py:75: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  return df[[value_col]].rename(columns={value_col: name}).resample("M").last()
/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_51876/1510097216.py:75: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  return df[[value_col]].rename(columns={value_col: name}).resample("M").last()
/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_51876/1510097216.py:75: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  return df[[value_col]].rename(